# Model selection bake-off

I am comparing three small instruction models on the same 40 manually audited development complaints. The run uses one zero-shot prompt and one fixed few-shot prompt, greedy decoding, 4-bit loading, and one MLflow run for each model and prompt pair.

In [1]:
!pip -q install 'transformers==5.10.1' 'peft==0.20.0' bitsandbytes accelerate 'mlflow==3.15.1'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 137.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import gc
import json
import os
import time
from importlib.metadata import version
from pathlib import Path

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from src.schema import (
    ALLOWED_DOMAINS,
    ALLOWED_ISSUES,
    SchemaError,
    validate_gold,
)

assert torch.cuda.is_available(), 'Select a Colab GPU runtime before running.'
drive.mount('/content/drive')
print('torch:', torch.__version__)
print('gpu:', torch.cuda.get_device_name(0))

MODEL_REVISION = 'main'
MAX_NEW_TOKENS = 192
BATCH_SIZE = 4
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/civicstruct-bakeoff')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_SPECS = [
    {'name': 'Qwen/Qwen3-4B-Instruct-2507', 'trust_remote_code': False},
    {'name': 'HuggingFaceTB/SmolLM3-3B', 'trust_remote_code': False},
]
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

Mounted at /content/drive
torch: 2.11.0+cu128
gpu: Tesla T4


In [3]:
data_paths = [
    Path('data/model_selection_development.jsonl'),
    Path('../data/model_selection_development.jsonl'),
]
data_path = next(path for path in data_paths if path.exists())
development = [json.loads(line) for line in data_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(development) == 40
for row in development:
    validate_gold(row['gold'])

demo_paths = [Path('data/gold_examples.jsonl'), Path('../data/gold_examples.jsonl')]
demo_path = next(path for path in demo_paths if path.exists())
static_demos = [json.loads(line) for line in demo_path.read_text(encoding='utf-8').splitlines() if line.strip()]
assert not {row['case_id'] for row in development} & {row['case_id'] for row in static_demos}
print('development complaints:', len(development))
print('fixed demonstrations:', len(static_demos))

development complaints: 40
fixed demonstrations: 5


In [4]:
SYSTEM_PROMPT = (
    'You structure public-service complaints. Return exactly one JSON object with these fields: '
    'service_domain, issue_type, location, event_date_or_time, amount_inr, service_identifier, '
    'urgency, missing_information, and formal_summary. Use null for an absent scalar fact. '
    'Use only the allowed labels from the task schema. Do not guess facts. Do not output '
    'reasoning or commentary.'
)

def build_messages(complaint, few_shot=False):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    if few_shot:
        for demo in static_demos:
            messages.extend([
                {'role': 'user', 'content': demo['complaint']},
                {'role': 'assistant', 'content': json.dumps(demo['gold'], separators=(',', ':'))},
            ])
    messages.append({'role': 'user', 'content': complaint})
    return messages

def load_candidate(spec):
    kwargs = {'revision': MODEL_REVISION, 'trust_remote_code': spec['trust_remote_code']}
    tokenizer = AutoTokenizer.from_pretrained(spec['name'], **kwargs)
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        spec['name'],
        **kwargs,
        quantization_config=quantization_config,
        device_map='auto',
    )
    model.eval()
    return tokenizer, model

def checkpoint_name(model_name, prompt_mode):
    safe_model = model_name.split('/')[-1].lower().replace('-', '_')
    return CHECKPOINT_ROOT / f'{safe_model}__{prompt_mode}.json'

def save_json(path, value):
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2), encoding='utf-8')
    temporary.replace(path)

def load_json(path, default):
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else default

def prepare_inputs(tokenizer, message_batch):
    template_args = {
        'tokenize': True,
        'add_generation_prompt': True,
        'return_dict': True,
        'return_tensors': 'pt',
        'padding': True,
        'enable_thinking': False,
    }
    try:
        return tokenizer.apply_chat_template(message_batch, **template_args)
    except TypeError:
        template_args.pop('enable_thinking')
        return tokenizer.apply_chat_template(message_batch, **template_args)

def generate_batch(tokenizer, model, rows, few_shot):
    messages = [build_messages(row['complaint'], few_shot) for row in rows]
    inputs = prepare_inputs(tokenizer, messages).to(next(model.parameters()).device)
    prompt_tokens = inputs['input_ids'].shape[-1]
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    responses = tokenizer.batch_decode(generated[:, prompt_tokens:], skip_special_tokens=True)
    return [
        {'case_id': row['case_id'], 'response': response.strip(), 'latency_seconds': round(elapsed / len(rows), 4), 'batch_size': len(rows)}
        for row, response in zip(rows, responses)
    ]

def generate_with_fallback(tokenizer, model, rows, few_shot):
    try:
        return generate_batch(tokenizer, model, rows, few_shot)
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower() or len(rows) == 1:
            raise
        torch.cuda.empty_cache()
        return [item for row in rows for item in generate_batch(tokenizer, model, [row], few_shot)]

In [5]:
def macro_f1(gold_values, predicted_values, labels):
    scores = []
    for label in labels:
        true_positive = sum(gold == label and predicted == label for gold, predicted in zip(gold_values, predicted_values))
        false_positive = sum(gold != label and predicted == label for gold, predicted in zip(gold_values, predicted_values))
        false_negative = sum(gold == label and predicted != label for gold, predicted in zip(gold_values, predicted_values))
        denominator = 2 * true_positive + false_positive + false_negative
        scores.append(0.0 if denominator == 0 else 2 * true_positive / denominator)
    return sum(scores) / len(scores)

def score_outputs(outputs):
    valid = 0
    gold_domains, predicted_domains = [], []
    gold_issues, predicted_issues = [], []
    missing_f1_parts = []
    hallucinated = 0
    filled = 0
    for row, output in zip(development, outputs):
        try:
            predicted = json.loads(output['response'])
            validate_gold(predicted)
            valid += 1
        except (json.JSONDecodeError, SchemaError, TypeError, ValueError):
            predicted = {}
        gold = row['gold']
        gold_domains.append(gold['service_domain'])
        predicted_domains.append(predicted.get('service_domain'))
        gold_issues.append(gold['issue_type'])
        predicted_issues.append(predicted.get('issue_type'))
        gold_missing = set(gold['missing_information'])
        predicted_missing = set(predicted.get('missing_information') or [])
        missing_f1_parts.append((gold_missing, predicted_missing))
        complaint = row['complaint'].lower()
        for field in ('location', 'event_date_or_time', 'service_identifier', 'amount_inr'):
            value = predicted.get(field)
            if value is None:
                continue
            filled += 1
            if str(value).lower().replace(',', '') not in complaint.replace(',', ''):
                hallucinated += 1
    missing_tp = sum(len(gold & predicted) for gold, predicted in missing_f1_parts)
    missing_fp = sum(len(predicted - gold) for gold, predicted in missing_f1_parts)
    missing_fn = sum(len(gold - predicted) for gold, predicted in missing_f1_parts)
    missing_denominator = 2 * missing_tp + missing_fp + missing_fn
    return {
        'schema_validity_rate': valid / len(outputs),
        'service_domain_macro_f1': macro_f1(gold_domains, predicted_domains, ALLOWED_DOMAINS),
        'issue_type_macro_f1': macro_f1(gold_issues, predicted_issues, ALLOWED_ISSUES),
        'missing_information_f1': 0.0 if missing_denominator == 0 else 2 * missing_tp / missing_denominator,
        'hallucinated_field_rate': 0.0 if filled == 0 else hallucinated / filled,
    }

## Clean reload check

Restart the runtime after the earlier smoke run, run this cell once, and keep its output with the bake-off record. It must finish before the comparison loop.

In [6]:
reload_tokenizer, reload_model = load_candidate(MODEL_SPECS[0])
print('clean reload:', MODEL_SPECS[0]['name'])
print('model class:', type(reload_model).__name__)
del reload_tokenizer, reload_model
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

clean reload: Qwen/Qwen3-4B-Instruct-2507
model class: Qwen3ForCausalLM


In [8]:
os.environ['MLFLOW_ALLOW_FILE_STORE'] = 'true'
import mlflow

MLFLOW_ROOT = CHECKPOINT_ROOT / 'mlruns'
MLFLOW_ROOT.mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri(MLFLOW_ROOT.as_uri())
mlflow.set_experiment('civicstruct-model-bakeoff')
summary_path = CHECKPOINT_ROOT / 'bakeoff_summary.json'
summary = load_json(summary_path, {'results': [], 'setup_failures': []})
all_results = summary['results']
setup_failures = summary['setup_failures']
completed_pairs = {(item['model_name'], item['prompt_mode']) for item in all_results}
for spec in MODEL_SPECS:
    try:
        tokenizer, model = load_candidate(spec)
        tokenizer.padding_side = 'left'
        if tokenizer.pad_token_id is None:
            tokenizer.pad_token = tokenizer.eos_token
    except Exception as error:
        failure = {'model_name': spec['name'], 'error_type': type(error).__name__, 'error': str(error)}
        setup_failures = [item for item in setup_failures if item['model_name'] != spec['name']] + [failure]
        save_json(summary_path, {'results': all_results, 'setup_failures': setup_failures})
        print(json.dumps(failure))
        continue
    for prompt_mode in ('zero_shot', 'static_few_shot'):
        pair = (spec['name'], prompt_mode)
        if pair in completed_pairs:
            print('already complete:', pair)
            continue
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
        checkpoint_path = checkpoint_name(spec['name'], prompt_mode)
        checkpoint = load_json(checkpoint_path, {'model_name': spec['name'], 'prompt_mode': prompt_mode, 'outputs': []})
        outputs_by_id = {item['case_id']: item for item in checkpoint['outputs']}
        remaining = [row for row in development if row['case_id'] not in outputs_by_id]
        for start in range(0, len(remaining), BATCH_SIZE):
            batch = remaining[start:start + BATCH_SIZE]
            generated = generate_with_fallback(tokenizer, model, batch, prompt_mode == 'static_few_shot')
            outputs_by_id.update({item['case_id']: item for item in generated})
            ordered_outputs = [outputs_by_id[row['case_id']] for row in development if row['case_id'] in outputs_by_id]
            save_json(checkpoint_path, {
                'model_name': spec['name'],
                'prompt_mode': prompt_mode,
                'model_revision': MODEL_REVISION,
                'batch_size': BATCH_SIZE,
                'max_new_tokens': MAX_NEW_TOKENS,
                'outputs': ordered_outputs,
            })
            print(f'{spec["name"]} {prompt_mode}: saved {len(ordered_outputs)}/{len(development)}')
        outputs = [outputs_by_id[row['case_id']] for row in development]
        scores = score_outputs(outputs)
        result = {
            'model_name': spec['name'],
            'model_revision': getattr(model.config, '_commit_hash', None) or MODEL_REVISION,
            'prompt_mode': prompt_mode,
            'device': 'cuda',
            'decoding': {'do_sample': False, 'max_new_tokens': MAX_NEW_TOKENS},
            'batch_size': BATCH_SIZE,
            'mean_latency_seconds_per_case': sum(item['latency_seconds'] for item in outputs) / len(outputs),
            'gpu_memory_mb': torch.cuda.max_memory_allocated() / 2**20,
            'package_versions': {name: version(name) for name in ('torch', 'transformers', 'mlflow')},
            'scores': scores,
        }
        try:
            with mlflow.start_run(run_name=spec['name'].split('/')[-1] + '-' + prompt_mode) as run:
                mlflow.log_params({'model_name': spec['name'], 'model_revision': result['model_revision'], 'prompt_mode': prompt_mode, 'decoding': 'greedy', 'batch_size': BATCH_SIZE})
                mlflow.log_metrics({**scores, 'mean_latency_seconds_per_case': result['mean_latency_seconds_per_case'], 'gpu_memory_mb': result['gpu_memory_mb']})
                mlflow.log_text(json.dumps(result, indent=2), 'run_metadata.json')
                mlflow.log_text(json.dumps(outputs, indent=2), 'outputs.json')
                result['mlflow_run_id'] = run.info.run_id
        except Exception as error:
            result['mlflow_error'] = f'{type(error).__name__}: {error}'
        all_results = [item for item in all_results if (item['model_name'], item['prompt_mode']) != pair] + [result]
        completed_pairs.add(pair)
        save_json(summary_path, {'results': all_results, 'setup_failures': setup_failures})
        print(json.dumps(result, indent=2))
    del tokenizer, model
    gc.collect()
    torch.cuda.empty_cache()

save_json(summary_path, {'results': all_results, 'setup_failures': setup_failures})
print(json.dumps({'results': all_results, 'setup_failures': setup_failures}, indent=2))

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 4/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 8/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 12/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 16/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 20/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 24/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 28/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 32/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 36/40
Qwen/Qwen3-4B-Instruct-2507 zero_shot: saved 40/40
{
  "model_name": "Qwen/Qwen3-4B-Instruct-2507",
  "model_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "prompt_mode": "zero_shot",
  "device": "cuda",
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 192
  },
  "batch_size": 4,
  "mean_latency_seconds_per_case": 3.6690599999999995,
  "gpu_memory_mb": 4604.89013671875,
  "package_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.10.1",
    "mlflow": "3.15.1"
  },
  "scores": {
    "schema_validity_r

config.json:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


HuggingFaceTB/SmolLM3-3B zero_shot: saved 4/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 8/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 12/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 16/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 20/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 24/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 28/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 32/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 36/40
HuggingFaceTB/SmolLM3-3B zero_shot: saved 40/40
{
  "model_name": "HuggingFaceTB/SmolLM3-3B",
  "model_revision": "a07cc9a04f16550a088caea529712d1d335b0ac1",
  "prompt_mode": "zero_shot",
  "device": "cuda",
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 192
  },
  "batch_size": 4,
  "mean_latency_seconds_per_case": 3.1785,
  "gpu_memory_mb": 2058.50732421875,
  "package_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.10.1",
    "mlflow": "3.15.1"
  },
  "scores": {
    "schema_validity_rate": 0.0,
    "service_domain_macro_f1": 0.0